In [10]:
%matplotlib qt5
import hyperspy.api as hs
import pyxem as pxm
import numpy as np
import orix
import matplotlib.pyplot as plt
import matplotlib.projections

from pathlib import Path
from diffpy.structure import Atom, Lattice, Structure
from diffsims.generators.simulation_generator import SimulationGenerator
from orix.vector import Miller, Vector3d
from orix.plot import IPFColorKeyTSL

In [2]:
PATH_SPED5 = Path("D:/datasets/250225_Ag_oxww/20250225_175525/SPED5_calibrated.zspy")
# PATH_SPED6 = Path("D:/datasets/250225_Ag_oxww/20250225_180346/SPED6_calibrated.zspy")
# PATH_SPED7 = Path("D:/datasets/250225_Ag_oxww/20250225_181109/SPED7_calibrated.zspy")
# PATH_SPED8 = Path("D:/datasets/250225_Ag_oxww/20250225_182247/SPED8_calibrated.zspy")

SPED5 = hs.load(PATH_SPED5, lazy=True)
# SPED6 = hs.load(PATH_SPED6, lazy=True)
# SPED7 = hs.load(PATH_SPED7, lazy=True)
# SPED8 = hs.load(PATH_SPED8, lazy=True)

# SPED5.change_dtype("float32")
# SPED6.change_dtype("float32")
# SPED7.change_dtype("float32")
# SPED8.change_dtype("float32")


ny, nx = SPED5.axes_manager.signal_shape
center = (ny/2 - 0.5, nx/2 - 0.5)

SPED5.calibration(center=center)
# SPED6.calibration(center=center)
# SPED7.calibration(center=center)
# SPED8.calibration(center=center)
# SPEDtest.calibration(center=center)

SPED5_template = SPED5.template_match_disk(disk_r=3, subtract_min=False)
# SPED6_template = SPED6.template_match_disk(disk_r=3, subtract_min=False)
# SPED7_template = SPED7.template_match_disk(disk_r=3, subtract_min=False)
# SPED8_template = SPED8.template_match_disk(disk_r=3, subtract_min=False)
# SPEDtest_template = SPEDtest.template_match_disk(disk_r=3, subtract_min=False)

SPED5_peaks = SPED5_template.get_diffraction_vectors(min_distance=3, threshold_abs=0.6)
# SPED6_peaks = SPED6_template.get_diffraction_vectors(min_distance=3, threshold_abs=0.6)
# SPED7_peaks = SPED7_template.get_diffraction_vectors(min_distance=3, threshold_abs=0.6)
# SPED8_peaks = SPED8_template.get_diffraction_vectors(min_distance=3, threshold_abs=0.6)
# SPEDtest_peaks = SPEDtest_template.get_diffraction_vectors(min_distance=3, threshold_abs=0.6)

# SPED5_markers = SPED5_peaks.to_markers(sizes=3, color='cyan')
# SPED6_markers = SPED6_peaks.to_markers(sizes=3, color='cyan')
# SPED7_markers = SPED7_peaks.to_markers(sizes=3, color='cyan')
# SPED8_markers = SPED8_peaks.to_markers(sizes=3, color='cyan')

SPED5_mask = SPED5_peaks.to_mask(disk_r=3)
# SPED6_mask = SPED6_peaks.to_mask(disk_r=3)
# SPED7_mask = SPED7_peaks.to_mask(disk_r=3)
# SPED8_mask = SPED8_peaks.to_mask(disk_r=3)
# SPEDtest_mask = SPEDtest_peaks.to_mask(disk_r=3)

SPED5_masked = SPED5 * SPED5_mask
# SPED6_masked = SPED6 * SPED6_mask
# SPED7_masked = SPED7 * SPED7_mask
# SPED8_masked = SPED8 * SPED8_mask
# SPEDtest_masked = SPEDtest * SPEDtest_mask

WARNING | Hyperspy | The function you applied does not take into account the difference of scales in-between axes. (hyperspy.signal:5597)
WARNING | Hyperspy | The function you applied does not take into account the difference of units in-between axes. (hyperspy.signal:5602)


In [3]:
SPED5_azi = SPED5_masked.get_azimuthal_integral2d(npt=256)
# SPED6_azi = SPED6_masked.get_azimuthal_integral2d(npt=256)
# SPED7_azi = SPED7_masked.get_azimuthal_integral2d(npt=256)
# SPED8_azi = SPED8_masked.get_azimuthal_integral2d(npt=256)

In [4]:
a = 4.08
atoms = [Atom('Ag', [0, 0, 0]), Atom('Ag', [0.5, 0.5, 0]), Atom('Ag', [0.5, 0, 0.5]), Atom('Ag', [0, 0.5, 0.5])]
lattice = Lattice(a, a, a, 90, 90, 90)
phase = orix.crystal_map.Phase(name='Ag', space_group=225, structure=Structure(atoms, lattice))

angular_resolution = 0.5
grid = orix.sampling.get_sample_reduced_fundamental(angular_resolution, point_group=phase.point_group)
orientations = orix.quaternion.Orientation(grid, symmetry=phase.point_group)

simgen = SimulationGenerator(precession_angle=1., minimum_intensity=1e-5)
simulations = simgen.calculate_diffraction2d(phase=phase, rotation=grid, reciprocal_radius=1.5, with_direct_beam=False, max_excitation_error=0.01)

In [5]:
SPED5_results = SPED5_azi.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)
# SPED6_results = SPED6_azi.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)
# SPED7_results = SPED7_azi.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)
# SPED8_results = SPED8_azi.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)

WARNING | Hyperspy | The function you applied does not take into account the difference of scales in-between axes. (hyperspy.signal:5597)
WARNING | Hyperspy | The function you applied does not take into account the difference of units in-between axes. (hyperspy.signal:5602)


In [6]:
SPED5_results.compute()
SPED5_xmap = SPED5_results.to_crystal_map()

  0%|          | 0/19 [00:00<?, ?it/s]

  0%|          | 0/65 [00:00<?, ?it/s]

# SPED5

In [ ]:
SPED5_results.compute()
SPED5_xmap = SPED5_results.to_crystal_map()
SPED5_oris = SPED5_xmap.orientations
SPED5_corrs = SPED5_results.data[:,:,0,1].flatten()

key_x = IPFColorKeyTSL(phase.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(phase.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(phase.point_group, Vector3d.zvector())

oris_z = key_z.orientation2color(SPED5_oris)
SPED5_xmap.plot(oris_z, overlay=SPED5_corrs, remove_padding=True)
oris_x = key_x.orientation2color(SPED5_oris)
SPED5_xmap.plot(oris_x, overlay=SPED5_corrs, remove_padding=True)
oris_y = key_y.orientation2color(SPED5_oris)
SPED5_xmap.plot(oris_y, overlay=SPED5_corrs, remove_padding=True)

In [16]:
ix, iy = SPED5.axes_manager.indices
SPED5.plot()

In [44]:
correlations_i = SPED5_results.inav[ix, iy].data[:, 1]
template_indices_i = (SPED5_results.inav[ix, iy].data[:, 0]).astype('int16')
orientations_i = orientations[template_indices_i]


fig = plt.figure()
ax = fig.add_subplot(111, projection="ipf", symmetry=phase.point_group)
ax.scatter(orientations_i, c=correlations_i, cmap='inferno')

# SPED6

In [ ]:
SPED6_results.compute()
SPED6_xmap = SPED6_results.to_crystal_map()
SPED6_oris = SPED6_xmap.orientations
SPED6_corrs = SPED6_results.data[:,:,0,1].flatten()

key_x = IPFColorKeyTSL(phase.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(phase.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(phase.point_group, Vector3d.zvector())

oris_z = key_z.orientation2color(SPED6_oris)
SPED6_xmap.plot(oris_z, overlay=SPED6_corrs, remove_padding=True)
oris_x = key_x.orientation2color(SPED6_oris)
SPED6_xmap.plot(oris_x, overlay=SPED6_corrs, remove_padding=True)
oris_y = key_y.orientation2color(SPED6_oris)
SPED6_xmap.plot(oris_y, overlay=SPED6_corrs, remove_padding=True)

In [ ]:
ix, iy = 20, 10

correlations_i = SPED6_results.inav[ix, iy].data[:, 1]
template_indices_i = (SPED6_results.inav[ix, iy].data[:, 0]).astype('int16')
orientations_i = orientations[template_indices_i]

# SPED6_results.inav[ix, iy].data[0, 0].astype('int16')

fig = plt.figure()
ax = fig.add_subplot(111, projection="ipf", symmetry=phase.point_group)
ax.scatter(orientations_i, c=correlations_i, cmap='inferno')

# SPED7

In [ ]:
SPED7_results.compute()
SPED7_xmap = SPED7_results.to_crystal_map()
SPED7_oris = SPED7_xmap.orientations
SPED7_corrs = SPED7_results.data[:,:,0,1].flatten()

key_x = IPFColorKeyTSL(phase.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(phase.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(phase.point_group, Vector3d.zvector())

oris_z = key_z.orientation2color(SPED7_oris)
SPED7_xmap.plot(oris_z, overlay=SPED7_corrs, remove_padding=True)
oris_x = key_x.orientation2color(SPED7_oris)
SPED7_xmap.plot(oris_x, overlay=SPED7_corrs, remove_padding=True)
oris_y = key_y.orientation2color(SPED7_oris)
SPED7_xmap.plot(oris_y, overlay=SPED7_corrs, remove_padding=True)

In [ ]:
ix, iy = SPED7.axes_manager.indices
SPED7.plot()

In [ ]:
correlations_i = SPED7_results.inav[ix, iy].data[:, 1]
template_indices_i = (SPED7_results.inav[ix, iy].data[:, 0]).astype('int16')
orientations_i = orientations[template_indices_i]

fig = plt.figure()
ax = fig.add_subplot(111, projection="ipf", symmetry=phase.point_group)
ax.scatter(orientations_i, c=correlations_i, cmap='inferno')

# SPED8

In [ ]:
SPED8_results.compute()
SPED8_xmap = SPED8_results.to_crystal_map()
SPED8_oris = SPED8_xmap.orientations
SPED8_corrs = SPED8_results.data[:,:,0,1].flatten()

key_x = IPFColorKeyTSL(phase.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(phase.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(phase.point_group, Vector3d.zvector())

oris_z = key_z.orientation2color(SPED8_oris)
SPED8_xmap.plot(oris_z, overlay=SPED8_corrs, remove_padding=True)
oris_x = key_x.orientation2color(SPED8_oris)
SPED8_xmap.plot(oris_x, overlay=SPED8_corrs, remove_padding=True)
oris_y = key_y.orientation2color(SPED8_oris)
SPED8_xmap.plot(oris_y, overlay=SPED8_corrs, remove_padding=True)

# Misorientations

In [ ]:
def iterate_x(Orientations_list, x_max):
    pixel_misorientation_matrix = []
    pixel_misorientation_list = []
    for i in range(Orientations_list.shape[0]):
        # start a new row whenever we hit a multiple of x_max
        if i % x_max == 0 and i != 0:
            pixel_misorientation_matrix.append(pixel_misorientation_list)
            pixel_misorientation_list = []

        # if we are not at the last column in this row
        if i % x_max != x_max - 1 and i != Orientations_list.shape[0]:
            # if left, me, and right are identical orientations → put 0
            if (Orientations_list[i-1] == Orientations_list[i]
                and Orientations_list[i] == Orientations_list[i+1]):
                pixel_misorientation_list.append(0)
            else:
                # otherwise compute misorientation to pixel on the right
                pixel_misorientation_list.append(
                    Orientations_list[i].angle_with_outer(
                        Orientations_list[i+1], degrees=True
                    )[0][0]
                )
        else:
            # last pixel in the row, no right neighbour → 0
            pixel_misorientation_list.append(0)

    pixel_misorientation_matrix.append(pixel_misorientation_list)
    return pixel_misorientation_matrix

def iterate_y(Orientations_list, x_max, y_max, pixel_misorientation_matrix):
    for i in range(Orientations_list.shape[0]):
        if i + x_max < Orientations_list.shape[0]:
            # if at least one of (above, me, below) is different
            if (Orientations_list[i - x_max] != Orientations_list[i]
                or Orientations_list[i] != Orientations_list[i + x_max]):

                current_val = Orientations_list[i].angle_with_outer(
                    Orientations_list[i + x_max], degrees=True
                )[0][0]

                # pick the LARGEST of: "x-direction misori we already had"
                # vs "y-direction misori we just computed"
                if current_val > pixel_misorientation_matrix[i // x_max][i % x_max]:
                    pixel_misorientation_matrix[i // x_max][i % x_max] = current_val
    return pixel_misorientation_matrix

def grain_boundaries_misorientations(Orientations_list, x_max, y_max):
    pixel_misorientation_matrix = iterate_x(Orientations_list, x_max)
    pixel_misorientation_matrix = iterate_y(
        Orientations_list, x_max, y_max, pixel_misorientation_matrix
    )
    return np.array(pixel_misorientation_matrix)

In [ ]:
test = grain_boundaries_misorientations(SPED8_oris, 200, 100)
plt.imshow(test)

In [ ]:
import matplotlib.pyplot as plt
from orix.plot import IPFColorKeyTSL

# Suppose your orientation map is called:
# xmap = area_xmaps[i]  or  xmap = SPED8_xmap

# 1️⃣ Define the color key (e.g. IPF along the sample Z direction)
ckey = IPFColorKeyTSL(phase.point_group, Vector3d.xvector())  # cubic for FCC metals like Ag

# 2️⃣ Generate the RGB colors for each pixel
ipf_colors = ckey.orientation2color(orientations)

# 3️⃣ Plot the IPF-colored map once
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(ipf_colors, origin="lower")
ax.set_title("IPF-Z Color Map")
ax.set_xlabel("X (pixels)")
ax.set_ylabel("Y (pixels)")

plt.tight_layout()
plt.show()

# 4️⃣ (Optional) Show the IPF color key legend
ckey.plot()
plt.show()